In [ ]:
# scripts/1_prepare.py
"""
Assemble & clean:
- reproject & resample all rasters to a common grid (EPSG and resolution)
- compute NDVI from Sentinel2 (if S2 bands available)
- compute slope from DSM, plus other topo derivatives
- save processed rasters under data/processed/{tile}_{layer}.tif
"""
import os
from glob import glob
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np
from scipy import ndimage
import rasterio.enums
import warnings
warnings.filterwarnings("ignore")

# --- CONFIG ---
SRC_CRS = "EPSG:4326"        # adapter à ton jeu; prefer local UTM zone for métrique
DST_CRS = "EPSG:4326"        # si tu veux rester en lat/lon; sinon exemple "EPSG:32631"
TARGET_RES = 30              # 30m target resolution
DATA_RAW = "../data/raw"
DATA_PROC = "../data/processed"
os.makedirs(DATA_PROC, exist_ok=True)

# helper: reproject/resample raster to match a target grid (using first DSM as reference)
def reproject_to_template(src_path, dst_path, dst_crs, dst_transform, dst_width, dst_height, dst_res):
    with rasterio.open(src_path) as src:
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': dst_transform,
            'width': dst_width,
            'height': dst_height
        })
        arr = np.zeros((src.count, dst_height, dst_width), dtype=src.meta['dtype'])
        for i in range(1, src.count+1):
            reproject(
                source=rasterio.band(src, i),
                destination=arr[i-1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear if src.count==1 else Resampling.nearest
            )
        kwargs.update({'count': src.count})
        with rasterio.open(dst_path, 'w', **kwargs) as dst:
            dst.write(arr)

# Build a template grid from first DSM found
dsm_paths = glob(os.path.join(DATA_RAW, "DSM", "*.tif"))
if len(dsm_paths) == 0:
    raise SystemExit("Place DSM tuiles under data/raw/DSM/*.tif")
template = dsm_paths[0]
with rasterio.open(template) as tmp:
    # compute template transform and size for TARGET_RES (simplified)
    left, bottom, right, top = tmp.bounds.left, tmp.bounds.bottom, tmp.bounds.right, tmp.bounds.top
    # Compute width/height in metres roughly if using projected CRS; if CRS is geographic this step should use transform properly
    # We'll reuse tmp transform but set resolution to TARGET_RES by approximating pixels.
    # Better: choose an appropriate projected CRS (UTM) for accurate meter resolution.
    dst_crs = tmp.crs  # keep same for now
    # compute new transform with desired resolution
    # Note: Here we simply ensure consistency with tmp but in production compute accurate transform in meters
    dst_transform = tmp.transform
    dst_width = tmp.width
    dst_height = tmp.height

# Reproject all layers to this template (simple approach)
layers = {
    'DSM': os.path.join(DATA_RAW, "DSM"),
    'impervious': os.path.join(DATA_RAW, "impervious"),
    'canopy': os.path.join(DATA_RAW, "canopy"),
    'sentinel2': os.path.join(DATA_RAW, "sentinel2"),  # expect B4 (red), B8 (nir) or precomputed NDVI
    'landcover': os.path.join(DATA_RAW, "landcover"),
    'lidar_dtm': os.path.join(DATA_RAW, "lidar_dtm")
}

for name, folder in layers.items():
    if not os.path.isdir(folder):
        continue
    for src in glob(os.path.join(folder, "*.tif")):
        dst = os.path.join(DATA_PROC, f"{os.path.splitext(os.path.basename(src))[0]}_{name}.tif")
        print("Processing", src, "->", dst)
        try:
            reproject_to_template(src, dst, dst_crs, dst_transform, dst_width, dst_height, TARGET_RES)
        except Exception as e:
            print("Warning reproject failed:", e)

# Compute NDVI if Sentinel2 bands available (expects *_B04.tif and *_B08.tif in processed)
proc_files = glob(os.path.join(DATA_PROC, "*sentinel2.tif"))
for p in proc_files:
    base = os.path.splitext(os.path.basename(p))[0].replace("_sentinel2","")
    # naive: assume two files per tile: base_B04_sent... and base_B08_sent...
    b4 = p.replace("B08","B04") if "B08" in p else None
    b8 = p if "B08" in p else None
    if not b4 or not b8 or not os.path.exists(b4) or not os.path.exists(b8):
        continue
    with rasterio.open(b4) as r4, rasterio.open(b8) as r8:
        red = r4.read(1).astype('float32')
        nir = r8.read(1).astype('float32')
        ndvi = (nir - red) / (nir + red + 1e-6)
        meta = r4.meta.copy()
        meta.update({'dtype':'float32','count':1})
        out = os.path.join(DATA_PROC, base + "_NDVI.tif")
        with rasterio.open(out, 'w', **meta) as dst:
            dst.write(ndvi, 1)
        print("Written NDVI", out)

# Compute slope from DSM (simple finite differences)
dsm_proc_files = glob(os.path.join(DATA_PROC, "*_DSM.tif"))
for d in dsm_proc_files:
    with rasterio.open(d) as src:
        arr = src.read(1).astype('float32')
        # compute gradient using sobel or simple gradient
        dx = ndimage.sobel(arr, axis=1)
        dy = ndimage.sobel(arr, axis=0)
        slope = np.hypot(dx, dy)
        meta = src.meta.copy()
        meta.update({'dtype':'float32','count':1})
        out = d.replace("_DSM.tif","_slope.tif")
        with rasterio.open(out, 'w', **meta) as dst:
            dst.write(slope, 1)
        print("Wrote slope", out)

print("Preprocessing finished. Inspect data/processed/")